# COURSE: Deep Learning for Music Technology
## SECTION: Transformers for music
### EXERCISE E3: Telling the model what you want (1): how busy is the bar?
#### Builds on: E2d (baroque mini-GPT)

**Goal.** In E2d the bass was the condition. But the melodies were too plain: about 4 notes per bar, when Corelli writes about 6. The bass does not say how fast the violin should go, so the model has to guess.

Now we **tell** it. In every bar, between the bass and the melody, we write a **control token**: `~` and one digit, the number of notes that start in the melody bar. `~9` means *9 or more*.

```
E2d:  "_6"E,4 F,4 & z/2g/2f/2g/2 e/2d/2c/2B/2 A/2a/2g/2a/2 f/2e/2d/2c/2|
E3:   "_6"E,4 F,4 & ~9 z/2g/2f/2g/2 e/2d/2c/2B/2 A/2a/2g/2a/2 f/2e/2d/2c/2|
```

Nobody explains to the model what `~9` means. It learns it from the examples, because the number was always true. Then **we** write the number, and choose calm or lively.

| Part | What you do |
|---|---|
| 1 | Add the control token to the text |
| 2 | **Fine-tune** the E2d model on the new text (your first fine-tuning) |
| 3 | Use it: calm, lively, a curve |
| 4 | Measure: does it obey? What does it cost? |
| ★ | Extra: control + candidate selection |

**You need** the second E2d checkpoint, `e2d_baroque_gpt_6keys.pt`, in the E2d folder.

# Setup

In [ ]:
### import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import math, os, re, time, random, urllib.request

# for data visualization
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
from IPython.display import clear_output

In [ ]:
# use the GPU if there is one (Colab: Runtime > Change runtime type > T4 GPU)
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

In [ ]:
#@title Where to save your work
# Colab can disconnect at any time. Checkpoints saved to Google Drive survive that.
USE_DRIVE = True  #@param {type:"boolean"}

try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB and USE_DRIVE:
  from google.colab import drive
  drive.mount('/content/drive')
  SAVE_DIR = '/content/drive/MyDrive/music_transformers/E3'
else:
  SAVE_DIR = 'E3_files'
E2D_DIR = SAVE_DIR.replace('E3', 'E2d')          # where the E2d checkpoint and data are
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(E2D_DIR, exist_ok=True)
print('Files will be saved in', SAVE_DIR)

In [ ]:
#@title Helper 1: Corelli tables and Bach chorales → ABC (run me, no need to read)
# Teacher-side data code. It is the same code as E2_research_scripts/baroque_check/baroque_to_abc.py
import csv, glob, os, re, sys
from fractions import Fraction as Fr

FIFTHS = dict(zip('FCGDAEB', range(-1, 6)))
NAT = dict(C=0, D=2, E=4, F=5, G=7, A=9, B=11); LET = 'CDEFGAB'
MODES = {'': 0, 'm': -3, 'dor': -2, 'mix': -1}           # what E2's abc_key_signature understands

def sig_acc(k):                                          # number of sharps -> {letter: accidental}
    order = 'FCGDAEB'; a = dict.fromkeys(LET, 0)
    for l in (order[:k] if k > 0 else order[::-1][:-k]): a[l] += 1 if k > 0 else -1
    return a

def key_field(tonic_letter, tonic_acc, k):               # K: text whose signature is k, with the tonic when a mode fits
    t = FIFTHS[tonic_letter] + 7 * tonic_acc
    for mode, off in MODES.items():
        if t + off == k: return tonic_letter + {1: '#', -1: 'b', 0: ''}[tonic_acc] + mode
    return next(l + {1: '#', -1: 'b', 0: ''}[a] for l in FIFTHS for a in (0, 1, -1) if FIFTHS[l] + 7 * a == k)

def key_to_sharps(K):                                    # inverse of key_field, used to read the text back
    m = re.match(r'([A-G])([#b]?)(m|dor|mix)?$', K); t = FIFTHS[m.group(1)] + 7 * {'#': 1, 'b': -1, '': 0}[m.group(2)]
    return t + MODES[m.group(3) or '']

# ------------------------------------------------------------------ pitch and duration <-> text
def abc_pitch(letter, octave):                           # C4 -> 'C', C5 -> 'c', C3 -> 'C,'
    if octave >= 5: return letter.lower() + "'" * (octave - 5)
    return letter + ',' * (4 - octave)

def dur_text(d):                                         # d in eighth notes
    if d == 1: return ''
    if d.denominator == 1: return str(d.numerator)
    return (str(d.numerator) if d.numerator > 1 else '') + '/' + str(d.denominator)

ACC = {0: '=', 1: '^', -1: '_', 2: '^^', -2: '__'}
class Writer:                                            # one voice in one bar, with ABC's accidental rule
    def __init__(self, sig): self.sig, self.state = sig, {}
    def note(self, p, d, tie):
        if p is None: return 'z' + dur_text(d)
        L, alt, octv = p; key = (L, octv); cur = self.state.get(key, self.sig[L])
        acc = ACC[alt] if alt != cur else ''; self.state[key] = alt
        return acc + abc_pitch(L, octv) + dur_text(d) + ('-' if tie else '')

# ------------------------------------------------------------------ bars -> ABC
def bar_text(bar, sig, fig, beat):                       # bar = list of (dur in quarters, pitch or None, tie, figures)
    w = Writer(sig); out = ''; pos = Fr(0)
    for d, p, tie, f in bar:
        if out and pos % beat == 0: out += ' '
        if f and fig: out += '"_' + f + '"'
        out += w.note(p, d * 2, tie); pos += d
    return out

def to_abc(x, n):
    sig = sig_acc(x['k']); beat = x['beat']
    head = [f'X:{n}', f"T:{x['title']}", f"C:{x['composer']}", f"R:{x['R']}", f"M:{x['meter']}", 'L:1/8', f"K:{x['K']}"]
    bars = [bar_text(b, sig, True, beat) + ' & ' + bar_text(m, sig, False, beat) + '|' for b, m in x['bars']]
    bars[-1] = bars[-1][:-1] + '|]'
    return '\n'.join(head) + '\n' + '\n'.join(''.join(bars[i:i + 4]) for i in range(0, len(bars), 4)) + '\n'

# ------------------------------------------------------------------ ABC -> bars (used only to check)
TOK = re.compile(r'"([^"]*)"|(\^\^|__|\^|_|=)?([A-Ga-gz])([,\']*)(\d*)(/?)(\d*)(-?)|(\s+)')
def from_abc(text):
    head = {l[0]: l[2:] for l in text.split('\n') if re.match(r'^[A-Za-z]:', l)}
    sig = sig_acc(key_to_sharps(head['K'])); body = ''.join(l for l in text.split('\n') if not re.match(r'^[A-Za-z]:', l))
    out = []
    for bar in re.split(r'\|\]?', body)[:-1]:
        sides = bar.split(' & '); assert len(sides) == 2, bar
        voices = []
        for s in sides:
            segs = []; state = {}; pos = 0; fig = ''
            for m in TOK.finditer(s):
                if m.group(1) is not None: fig = m.group(1)[1:]; continue
                if m.group(9): continue
                num, sl, den = m.group(5), m.group(6), m.group(7)
                d = Fr(int(num) if num else 1, (int(den) if den else 2) if sl else 1)
                if m.group(3) == 'z': segs.append((d / 2, None, False, fig)); fig = ''; continue
                L = m.group(3).upper(); octv = (5 if m.group(3).islower() else 4) + m.group(4).count("'") - m.group(4).count(',')
                key = (L, octv)
                if m.group(2): state[key] = {'^': 1, '_': -1, '=': 0, '^^': 2, '__': -2}[m.group(2)]
                segs.append((d / 2, (L, state.get(key, sig[L]), octv), bool(m.group(8)), fig)); fig = ''
            voices.append(segs)
        out.append(tuple(voices))
    return head, out

# ------------------------------------------------------------------ shared helpers for the two loaders
def legal(d):                                            # quarters -> durations that have a note symbol (plain or dotted), longest first
    out = []
    while d > 0:
        best = None
        for k in range(-6, 4):
            for base in (Fr(1), Fr(3, 2)):
                v = base * Fr(2) ** k
                if v <= d and (best is None or v > best): best = v
        if best is None: out.append(d); break
        out.append(best); d -= best
    return out
def split_note(t, d, p, tie):                            # (onset, dur, pitch, tie) -> legal pieces; a piece is tied to the next one
    parts = legal(d); out, x = [], t
    for i, v in enumerate(parts):
        out.append((x, v, p, tie if i == len(parts) - 1 else (p is not None))); x += v
    return out

def fill(notes, length):                                 # notes: (onset, dur, pitch, tie, figs) inside one bar, sorted; gaps become rests
    out, t = [], Fr(0)
    for on, d, p, tie, f in notes:
        if on > t: out += [(v, None, False, '') for _, v, _, _ in split_note(Fr(0), on - t, None, False)]
        assert on >= t, 'overlapping notes'
        out.append((d, p, tie, f)); t = on + d
    if t < length: out += [(v, None, False, '') for _, v, _, _ in split_note(Fr(0), length - t, None, False)]
    return out, t

def cut_at_figures(notes, figs):                         # figs: list of (time, text); a figure inside a note splits it with a tie
    out = []
    for on, d, p, tie in [q for n in notes for q in split_note(*n)]:
        pts = [on] + sorted({t for t, f in figs if on < t < on + d})
        for i, s in enumerate(pts):
            e = pts[i + 1] if i + 1 < len(pts) else on + d
            f = next((f for t, f in figs if t == s), '') if p else ''
            out.append((s, e - s, p, (i + 1 < len(pts)) or tie, f))
    return out

def fig_text(tokens):                                    # source figure symbols -> text: h = natural, backslash = raised
    return ' '.join(t.replace('h', 'n').replace('\\', '+').replace('-', 'b').replace('_', '').replace('|', '').replace('r', '') for t in tokens).strip()

def beat_of(meter):
    num, den = (int(x) for x in meter.split('/'))
    return Fr(3, 2) * (Fr(4, den)) if num % 3 == 0 and num > 3 and den == 8 else Fr(4, den)

# ------------------------------------------------------------------ Corelli
NAME = re.compile(r'([A-G])(#{0,2}|b{0,2})(-?\d+)')
def pitch(name):
    m = NAME.match(name); return (m.group(1), m.group(2).count('#') - m.group(2).count('b'), int(m.group(3)))

def corelli(dir_):
    meta = {r['piece']: r for r in csv.DictReader(open(dir_ + '/metadata.tsv'), delimiter='\t')}
    out, skipped = [], []
    for name, md in meta.items():
        rd = lambda k: list(csv.DictReader(open(f'{dir_}/{k}/{name}.{k}.tsv'), delimiter='\t'))
        meas, notes, chords = rd('measures'), rd('notes'), rd('chords')
        ts = {m['timesig'] for m in meas}; ks = {m['keysig'] for m in meas}
        if len(ts) > 1 or len(ks) > 1: skipped.append((name, 'several meters or key signatures')); continue
        if any(Fr(n['duration_qb']).limit_denominator(24).denominator not in (1, 2, 4, 8, 16) for n in notes): skipped.append((name, 'triplets')); continue
        meter, k = ts.pop(), int(ks.pop())
        length = {int(m['mc']): Fr(m['act_dur']) * 4 for m in meas}
        staves = {n['staff'] for n in notes}; bass_st = '4' if '4' in staves else '3'
        figs = {}
        for c in chords:
            lv = [c[l] for l in sorted(x for x in c if x.startswith('thoroughbass_level')) if c[l]]
            t = fig_text([v for v in lv if v != '_'])
            if c['staff'] == bass_st and t: figs.setdefault(int(c['mc']), []).append((Fr(c['mc_onset']) * 4, t))
        def voice(staff):
            byb = {mc: [] for mc in length}
            for n in notes:
                if n['staff'] != staff or n['voice'] != '1': continue
                byb[int(n['mc'])].append((Fr(n['mc_onset']) * 4, Fr(n['duration_qb']).limit_denominator(24), pitch(n['name']), n['tied'] in ('1', '0')))
            return byb
        ann = md['annotated_key'].strip()
        K = key_field(ann[0].upper(), ann.count('#') - ann.count('b') if len(ann) > 1 else 0, k)
        qpm = next((float(c['qpm']) for c in chords if c.get('qpm')), None)
        for st, nm in (('1', 'violin 1'), ('2', 'violin 2')):
            if st not in staves: continue
            bv, mv = voice(bass_st), voice(st); bars = []
            for mc in sorted(length):
                b, _ = fill(cut_at_figures(sorted(bv[mc]), figs.get(mc, [])), length[mc])
                m, _ = fill([(on, d, p, tie, '') for n in sorted(mv[mc]) for on, d, p, tie in split_note(*n)], length[mc])
                bars.append((b, m))
            out.append(dict(id=f'{name}/{nm}', title=f"{md['workTitle']} {md['workNumber']}, {md['movementNumber']}. {md['movementTitle']} ({nm})",
                            composer='Arcangelo Corelli', R=md['movementTitle'].split(',')[0].lower() or 'movement', meter=meter, k=k, K=K, qpm=qpm, beat=beat_of(meter), bars=bars, group=name[:7]))
    return out, skipped

# ------------------------------------------------------------------ BCFB chorales
def kern_token(tok):                                      # (dur in quarters, (letter, alter, octave) or None, tie to next); None if unusable
    m = re.match(r'[\[\(\{]*(\d+)(\.*)', tok)
    if not m or 'q' in tok: return None
    d = Fr(4, int(m.group(1))) if int(m.group(1)) else None
    if d is None: return None
    add = d
    for _ in m.group(2): add /= 2; d += add
    p = re.search(r'([a-gA-G]+)(##|--|#|-|n)?', tok)
    if not p: return (d, None, False)
    L = p.group(1)[0].upper(); n = len(p.group(1)); a = {'#': 1, '##': 2, '-': -1, '--': -2}.get(p.group(2), 0)
    octv = 4 + (n - 1) if p.group(1)[0].islower() else 3 - (n - 1)
    return (d, (L, a, octv), ('[' in tok or '_' in tok))

def chorales(dir_):
    out, skipped = [], []
    for f in sorted(glob.glob(dir_ + '/*.krn')):
        name = os.path.basename(f).replace('_FB.musicxml.krn', '').replace('.krn', '')
        lines = [l for l in open(f, encoding='utf-8', errors='replace').read().split('\n') if l]
        head = next(l for l in lines if l.startswith('**')).split('\t')
        if head != ['**kern', '**fb', '**kern', '**kern', '**kern', '**kern']: skipped.append((name, 'not bass + figures + 4 voices')); continue
        ks, meters, clock, ev, figs, bars, ok = 0, set(), [Fr(0)] * 6, {s: [] for s in range(6)}, [], [], True
        for l in lines:
            p = l.split('\t')
            if l.startswith('!') or len(p) != 6: continue
            if p[0].startswith('*k['): ks = sum(1 if c == '#' else -1 for c in re.findall(r'#|-', p[0]))
            if re.match(r'\*M\d', p[0]): meters.add(p[0][2:])
            if l.startswith('='): bars.append(max(clock)); continue
            if l.startswith('*'): continue
            st = {s: kern_token(x) for s, x in enumerate(p) if s != 1 and x != '.'}
            if any(v is None for v in st.values()): ok = False; break
            t = min(clock[s] for s in st)
            if any(clock[s] != t for s in st): ok = False; break
            for s, (d, pitch_, tie) in st.items(): ev[s].append((t, d, pitch_, tie)); clock[s] = t + d
            if p[1] not in ('.', ''): figs.append((t, fig_text(p[1].split())))
        if not ok or len(meters) != 1 or len(set(clock[s] for s in (0, 2, 3, 4, 5))) != 1: skipped.append((name, 'grace notes, unequal voices or several meters')); continue
        starts = sorted(set(bars + [Fr(0)])); end = clock[0]; meter = meters.pop(); out_bars = []
        last = [p for t, d, p, tie in ev[0] if p][-1]
        for a, b in zip(starts, starts[1:] + [end]):
            if b <= a: continue
            def voice(s, with_figs):
                ns = []
                for t, d, p, tie in ev[s]:
                    if t < a or t >= b: continue
                    assert t + d <= b, (name, 'note crosses a bar')
                    ns.append((t - a, d, p, tie))
                if not with_figs: return fill([(on, d, p, tie, '') for n in ns for on, d, p, tie in split_note(*n)], b - a)[0]
                return fill(cut_at_figures(ns, [(x - a, tx) for x, tx in figs if a <= x < b] if with_figs else []), b - a)[0]
            out_bars.append((voice(0, True), voice(5, False)))
        out.append(dict(id=f'{name}', title=f'Chorale {name}', composer='J. S. Bach', R='chorale', meter=meter, k=ks,
                        K=key_field(last[0], last[1], ks), qpm=None, beat=beat_of(meter), bars=out_bars, group=name))
    return out, skipped

# ------------------------------------------------------------------ transposition (data augmentation)
def shift_pitch(p, f):                                    # f = steps on the circle of fifths (+1: C -> G). The shortest way up or down
    L, alt, octv = p; pos = FIFTHS[L] + 7 * alt + f
    L2 = 'FCGDAEB'[(pos + 1) % 7]; alt2 = (pos + 1) // 7
    m = 12 * (octv + 1) + NAT[L] + alt + (7 * f + 5) % 12 - 5
    return (L2, alt2, (m - NAT[L2] - alt2) // 12 - 1)

def shift_figures(fig, letter, sig0, letter2, sig1):      # 6, 7, 4 2... stay; a natural, sharp or flat may change its sign
    out = []
    for t in fig.split():
        m = re.search(r'\d', t); n = int(m.group()) if m else 3
        n = 2 if n == 9 else n
        a0, a1 = sig0[LET[(LET.index(letter) + n - 1) % 7]], sig1[LET[(LET.index(letter2) + n - 1) % 7]]
        sign = next((c for c in t if c in '#bn'), None)
        if sign and '+' not in t:
            delta = {'#': 1, 'b': -1, 'n': -a0}[sign]
            t = t.replace(sign, '' if delta == 0 else 'n' if a1 + delta == 0 else '#' if delta > 0 else 'b', 1) or '3'
        out.append(t)
    return ' '.join(out)

def transpose(tune, f):
    if f == 0: return tune
    head, bars = from_abc(tune); k = key_to_sharps(head['K']); sig0, sig1 = sig_acc(k), sig_acc(k + f); beat = beat_of(head['M'])
    t = FIFTHS[head['K'][0]] + 7 * ((head['K'][1:2] == '#') - (head['K'][1:2] == 'b')) + f
    K = key_field('FCGDAEB'[(t + 1) % 7], (t + 1) // 7, k + f); out = []
    for bass, mel in bars:
        b = []
        for d, p, tie, fig in bass:
            q = shift_pitch(p, f) if p else None
            b.append((d, q, tie, shift_figures(fig, p[0], sig0, q[0], sig1) if fig else ''))
        m = [(d, shift_pitch(p, f) if p else None, tie, '') for d, p, tie, _ in mel]
        out.append(bar_text(b, sig1, True, beat) + ' & ' + bar_text(m, sig1, False, beat) + '|')
    out[-1] = out[-1][:-1] + '|]'
    return '\n'.join(tune.split('\n')[:6] + ['K:' + K]) + '\n' + '\n'.join(''.join(out[i:i + 4]) for i in range(0, len(out), 4)) + '\n'

def nearby_keys(tune, n):                                 # the n nearest keys on the circle of fifths (itself included), at most 4 sharps or flats
    k = key_to_sharps(re.search(r'^K:(.*)$', tune, re.M).group(1))
    return sorted((f for f in range(-9, 10) if f == 0 or -4 <= k + f <= 4), key=lambda f: (abs(f), f))[:n]

# ------------------------------------------------------------------ check
def norm(bars):                                           # what must survive the text: pitches, rhythm, ties, figures, per voice and bar
    return [tuple(tuple((d, p, t, f) for d, p, t, f in v) for v in (b, m)) for b, m in bars]

def verify(x, text):
    head, got = from_abc(text); exp = norm(x['bars'])
    if len(got) != len(exp): return f'{len(got)} bars read, {len(exp)} expected'
    for i, (g, e) in enumerate(zip(norm(got), exp)):
        for v, name in ((0, 'bass'), (1, 'melody')):
            if g[v] != e[v]: return f'bar {i} {name}: {g[v]} != {e[v]}'
    return None

In [ ]:
#@title Helper 2: read, check, listen (run me, no need to read)
import json, re, uuid
from fractions import Fraction as Fr
from IPython.display import HTML, display

def split_abc(abc):
    """(header dict, body text) of one tune"""
    head, body = {}, []
    for line in abc.strip().split('\n'):
        m = re.match(r'^([A-Za-z]):(.*)$', line)
        if m and not body: head[m.group(1)] = m.group(2).strip()
        elif line.strip(): body.append(line)
    return head, '\n'.join(body)

def abc_body(abc): return split_abc(abc)[1]

def full_bar(meter):                       # length of a bar in quarter notes
    n, d = (int(x) for x in meter.split('/')); return Fr(4 * n, d)

# ---------------------------------------------------------------- tolerant reader (the model can write nonsense)
def read_side(s, sig):
    """one voice of one bar -> (list of (length in quarters, pitch or None, tie, figures), number of unreadable characters)"""
    segs, state, errors, fig, pos = [], {}, 0, '', 0
    while pos < len(s):
        m = TOK.match(s, pos)
        if not m or m.end() == m.start(): errors += 1; pos += 1; continue
        pos = m.end()
        if m.group(1) is not None:
            fig = m.group(1)[1:] if m.group(1)[:1] == '_' else ''; continue
        if m.group(9): continue
        d = Fr(int(m.group(5)) if m.group(5) else 1, (int(m.group(7)) if m.group(7) else 2) if m.group(6) else 1) / 2
        if m.group(3) == 'z': segs.append((d, None, False, fig)); fig = ''; continue
        L = m.group(3).upper(); octv = (5 if m.group(3).islower() else 4) + m.group(4).count("'") - m.group(4).count(',')
        if m.group(2): state[(L, octv)] = {'^': 1, '_': -1, '=': 0, '^^': 2, '__': -2}[m.group(2)]
        segs.append((d, (L, state.get((L, octv), sig[L]), octv), bool(m.group(8)), fig)); fig = ''
    return segs, errors

def read_bars(abc):
    """(header, bars): every bar is None if it does not have exactly one '&', else (bass, melody, unreadable characters)"""
    head, body = split_abc(abc)
    try: sig = sig_acc(key_to_sharps(head.get('K', 'C')))
    except Exception: sig = sig_acc(0)
    bars = []
    body = re.sub(r'~\d* ?', '', body)             # E3: the control token is not music
    for b in body.replace('\n', '').replace('|]', '|').split('|')[:-1]:
        sides = b.split('&')
        if len(sides) != 2: bars.append(None); continue
        (bass, e1), (mel, e2) = read_side(sides[0], sig), read_side(sides[1], sig)
        bars.append((bass, mel, e1 + e2))
    return head, bars

def length(segs): return sum((s[0] for s in segs), Fr(0))

def check_abc(abc):
    """n_bars, bar_ok (share of bars whose two voices have the same, right length; a first or last bar may be shorter),
       errors (unreadable characters, plus one for every bar without '&'), finished (ends with |])"""
    head, bars = read_bars(abc)
    try: full = full_bar(head.get('M', '4/4'))
    except Exception: full = Fr(4)
    ok = errors = 0
    for i, b in enumerate(bars):
        if b is None: errors += 1; continue
        a, m = length(b[0]), length(b[1]); errors += b[2]
        edge = i == 0 or i == len(bars) - 1
        ok += a == m and (a == full or (edge and 0 < a < full))
    return dict(n_bars=len(bars), bar_ok=ok / len(bars) if bars else 0.0, errors=errors, finished=abc.rstrip().endswith('|]'))

# ---------------------------------------------------------------- figured bass -> chord
def realize(letter, figs, sig, bass_pc):
    """pitch classes of the chord above a bass note (letter) with its figures ('6', '7', '#', '4 2', ...); no figure = a plain triad"""
    nums = {}
    for f in figs:
        m = re.search(r'\d', f); n = int(m.group()) if m else 3
        n = 2 if n == 9 else n
        pre, post = (f[:m.start()], f[m.start() + 1:]) if m else (f, '')
        nums[n] = '#' if any(c in pre + post for c in '#+') else 'b' if 'b' in pre + post else 'n' if 'n' in pre + post else ''
    if not nums: nums = {3: '', 5: ''}
    s = set(nums)
    if s in ({4, 2}, {2}): nums.setdefault(6, '')
    if s == {6}: nums[3] = ''
    if s in ({3}, {5}): nums.setdefault(3, ''); nums.setdefault(5, '')
    if s == {4}: nums[5] = ''
    if s in ({7}, {2, 7}): nums.setdefault(3, ''); nums.setdefault(5, '')
    if s == {6, 5}: nums[3] = ''
    if s == {4, 3}: nums[6] = ''
    i, out = LET.index(letter), {bass_pc}
    for n, acc in nums.items():
        L = LET[(i + n - 1) % 7]
        out.add((NAT[L] + (0 if acc == 'n' else sig[L] + {'#': 1, 'b': -1, '': 0}[acc])) % 12)
    return out

def chord_list(bass, sig):
    """for each bass segment: (start, length, pitch classes or None); a tied segment without figures keeps the chord before it"""
    out, t, prev, tied = [], Fr(0), None, False
    for d, p, tie, f in bass:
        if p is None: pcs = None
        elif tied and not f and prev is not None: pcs = prev
        else: pcs = realize(p[0], f.split(), sig, (NAT[p[0]] + p[1]) % 12)
        out.append((t, d, pcs)); prev = pcs; tied = tie; t += d
    return out

def strong_fit(abc):
    """share of strong-beat melody notes (beats 1 and 3 of 4/4 and 2/2, beat 1 of other meters) in the chord of bass + figures;
       the same with the chord moved up 5 and 7 semitones (what you would get by chance). Returns (fit %, moved %, number of notes)"""
    head, bars = read_bars(abc)
    try: sig = sig_acc(key_to_sharps(head.get('K', 'C')))
    except Exception: sig = sig_acc(0)
    strong = {Fr(0), Fr(2)} if head.get('M') in ('4/4', '2/2') else {Fr(0)}
    hit = moved = n = 0
    for b in bars:
        if b is None: continue
        t = Fr(0)
        for d, p, tie, f in b[1]:
            for s in strong:
                if t <= s < t + d and p is not None:
                    ch = next((c for a, l, c in chord_list(b[0], sig) if a <= s < a + l), None)
                    if ch is None: continue
                    pc = (NAT[p[0]] + p[1]) % 12; n += 1; hit += pc in ch
                    moved += sum(pc in {(x + k) % 12 for x in ch} for k in (5, 7)) / 2
            t += d
    return (100 * hit / n, 100 * moved / n, n) if n else (float('nan'), float('nan'), 0)

# ---------------------------------------------------------------- sheet music + audio (abcjs)
_ABCJS = "https://cdn.jsdelivr.net/npm/abcjs@6.4.4/dist/abcjs-basic-min.js"
_ABCJS_CSS = "https://cdn.jsdelivr.net/npm/abcjs@6.4.4/abcjs-audio.css"
_TEMPO = {'grave': 50, 'adagio': 60, 'largo': 56, 'andante': 80, 'allegro': 110, 'vivace': 120, 'presto': 140, 'chorale': 72}

def midi_abc(m):                           # MIDI number -> ABC with a sharp sign when it is a black key (audio only)
    name = ['C', '^C', 'D', '^D', 'E', 'F', '^F', 'G', '^G', 'A', '^A', 'B'][m % 12]
    L = name[-1]; o = m // 12 - 1
    return name[:-1] + abc_pitch(L, o)

def score_abc(abc):
    """the tune as two staves (what a musician reads)"""
    head, bars = read_bars(abc); bars = [b for b in bars if b is not None]
    try: sig, beat = sig_acc(key_to_sharps(head.get('K', 'C'))), beat_of(head.get('M', '4/4'))
    except Exception: sig, beat = sig_acc(0), Fr(1)
    H = f"X:1\nT:{head.get('T', '')}\nC:{head.get('C', '')}\nM:{head.get('M', '4/4')}\nL:1/8\n%%score {{M B}}\nK:{head.get('K', 'C')}\nV:M clef=treble name=\"Violin\"\nV:B clef=bass name=\"Continuo\"\n"
    out = []
    for i in range(0, len(bars), 4):
        for v, k in ((1, 'M'), (0, 'B')):
            out.append(f'[V:{k}] ' + '|'.join(bar_text(b[v], sig, v == 0, beat) for b in bars[i:i + 4]) + '|')
    return H + '\n'.join(out)

def play_bar(segs):                        # audio only: an accidental on every note (abcjs 6.4.4 forgets accidentals after a tie over the bar line)
    return ' '.join('z' + dur_text(d * 2) if p is None else ACC[p[1]] + abc_pitch(p[0], p[2]) + dur_text(d * 2) + ('-' if tie else '') for d, p, tie, f in segs)

def play_abc(abc, tempo, chords=True):
    """the same with instruments: violin, cello and a harpsichord chord on every bass note (audio only)"""
    head, bars = read_bars(abc); bars = [b for b in bars if b is not None]
    try: sig = sig_acc(key_to_sharps(head.get('K', 'C')))
    except Exception: sig = sig_acc(0)
    H = f"X:1\nM:{head.get('M', '4/4')}\nL:1/8\nQ:1/4={tempo}\n%%score {'M B H' if chords else 'M B'}\nK:{head.get('K', 'C')}\nV:M\n%%MIDI program 40\nV:B\n%%MIDI program 42\n" + ("V:H\n%%MIDI program 6\n" if chords else '')
    out = []
    for b in bars:
        txt = {}
        for v, k in ((1, 'M'), (0, 'B')): txt[k] = play_bar(b[v])
        ch = []
        for a, l, pcs in chord_list(b[0], sig):
            if pcs is None: ch.append('z' + dur_text(l * 2)); continue
            ch.append('[' + ''.join(midi_abc(next(m for m in range(55, 67) if m % 12 == pc)) for pc in sorted(pcs)) + ']' + dur_text(l * 2))
        txt['H'] = ' '.join(ch)
        for k in ('MBH' if chords else 'MB'): out.append(f'[V:{k}] {txt[k]}|')
    return H + '\n'.join(out)

def show_abc(abc, tempo=None, midi_name='tune', chords=None):
    """sheet music with a play button and a MIDI download link (abcjs, runs in your browser).
       chords: play the harpsichord chords? None = what PLAY_CHORDS says"""
    head, bars = read_bars(abc)
    if not any(bars): print('nothing readable to show'); return
    tempo = tempo or next((v for k, v in _TEMPO.items() if k in head.get('R', '').lower()), 90)
    uid = uuid.uuid4().hex[:8]
    display(HTML(f"""
<link rel="stylesheet" href="{_ABCJS_CSS}">
<div id="abc-{uid}" style="background:white;color:black"></div>
<div id="audio-{uid}" style="max-width:600px"></div>
<div id="midi-{uid}" style="font-size:13px;margin:4px 0"></div>
<div id="warn-{uid}" style="color:#b00;font-size:12px"></div>
<script>
(function() {{
  var abc = {json.dumps(score_abc(abc))}, audio = {json.dumps(play_abc(abc, tempo, PLAY_CHORDS if chords is None else chords))};
  function go() {{
    var tune = ABCJS.renderAbc("abc-{uid}", abc, {{responsive: "resize", add_classes: true}})[0];
    var w = tune.warnings || [];
    document.getElementById("warn-{uid}").innerHTML = w.length ? ("ABC warnings: " + w.length + "<br>" + w.slice(0, 5).join("<br>")) : "";
    var played = ABCJS.parseOnly(audio)[0];
    var opts = {{chordsOff: true}};
    if (ABCJS.synth.supportsAudio()) {{
      var ctl = new ABCJS.synth.SynthController();
      ctl.load("#audio-{uid}", null, {{displayPlay: true, displayProgress: true, displayRestart: true, displayLoop: true}});
      ctl.setTune(played, false, opts);
    }}
    var link = ABCJS.synth.getMidiFile(played, Object.assign({{midiOutputType: "link", fileName: "{midi_name}"}}, opts));
    document.getElementById("midi-{uid}").innerHTML = "&#11015; " + link;
  }}
  if (window.ABCJS) {{ go(); }} else {{
    var s = document.createElement("script"); s.src = "{_ABCJS}"; s.onload = go;
    document.head.appendChild(s);
  }}
}})();
</script>"""))

def make_prompt(title='Sonata da chiesa op. 9, no. 1, 2. Allegro (violin 1)', composer='Arcangelo Corelli', kind='allegro', meter='4/4', key='F'):
    """the header of a tune: whatever you put here is what the model continues"""
    return f'X:1\nT:{title}\nC:{composer}\nR:{kind}\nM:{meter}\nL:1/8\nK:{key}\n'

def report(tune):
    c = check_abc(tune); f = strong_fit(tune)
    print(f"{c['n_bars']} bars · {100*c['bar_ok']:.0f}% with the right length · {c['errors']} unreadable · finished: {c['finished']} · "
          f"strong-beat fit {f[0]:.0f}% (chance {f[1]:.0f}%)")

In [ ]:
#@title Listening
PLAY_CHORDS = True   #@param {type:"boolean"}
# True: violin, cello and harpsichord chords. False: only violin and cello. After a change, run this cell and then the cell you want to hear.
# For one tune only: show_abc(tune, chords=False)

In [ ]:
#@title Helper 3: the model, training and sampling of E2d (run me, no need to read)
class CausalSelfAttention(nn.Module):
  # the self_attention function above, with n_head heads at once and learnable weights
  def __init__(self, n_embd, n_head, block_size, dropout):
    super().__init__()
    self.n_head = n_head
    self.qkv  = nn.Linear(n_embd, 3*n_embd)      # Wq, Wk and Wv in one matrix
    self.proj = nn.Linear(n_embd, n_embd)        # mixes the heads back together
    self.drop = nn.Dropout(dropout)
    self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)).bool())
    self.att = None                              # we keep the last attention weights for Part 6

  def forward(self, x):
    B, T, C = x.shape                            # batch, time (characters), channels
    q, k, v = self.qkv(x).split(C, dim=2)
    # split the channels into heads: (B, n_head, T, C/n_head)
    q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
    k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
    v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
    scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
    att = F.softmax(scores, dim=-1)
    self.att = att.detach()
    y = self.drop(att) @ v
    y = y.transpose(1, 2).reshape(B, T, C)       # glue the heads back together
    return self.proj(y)


class Block(nn.Module):
  # attention (look around) + MLP (think about it), each with LayerNorm and a residual connection
  def __init__(self, n_embd, n_head, block_size, dropout):
    super().__init__()
    self.ln1  = nn.LayerNorm(n_embd)
    self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
    self.ln2  = nn.LayerNorm(n_embd)
    self.mlp  = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                              nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

  def forward(self, x):
    x = x + self.attn(self.ln1(x))
    x = x + self.mlp(self.ln2(x))
    return x


class MusicGPT(nn.Module):
  def __init__(self, vocab_size, block_size=256, n_embd=256, n_head=4, n_layer=4, dropout=0.1, use_pos=True):
    super().__init__()
    self.config = dict(vocab_size=vocab_size, block_size=block_size, n_embd=n_embd,
                       n_head=n_head, n_layer=n_layer, dropout=dropout, use_pos=use_pos)
    self.block_size = block_size
    self.use_pos = use_pos
    self.tok_emb = nn.Embedding(vocab_size, n_embd)      # which character?
    self.pos_emb = nn.Embedding(block_size, n_embd)      # where in the context?
    self.drop    = nn.Dropout(dropout)
    self.blocks  = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
    self.ln_f    = nn.LayerNorm(n_embd)
    self.head    = nn.Linear(n_embd, vocab_size)         # a score for every possible next character

  def forward(self, idx):
    B, T = idx.shape
    x = self.tok_emb(idx)                                # (B, T, n_embd)
    if self.use_pos:
      x = x + self.pos_emb(torch.arange(T, device=idx.device))
    x = self.blocks(self.drop(x))
    return self.head(self.ln_f(x))                       # (B, T, vocab_size)

def get_batch(data, batch_size=32, block_size=512):
  # random starting points, then input x and target y (shifted version of x, as in loremipsum)
  ix = torch.randint(len(data) - block_size - 1, (batch_size,))
  x = torch.stack([data[i   : i+block_size  ] for i in ix])
  y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
  return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, n_batches=20, batch_size=16):
  # average loss over some random batches, with dropout switched off
  model.eval()
  losses = []
  for _ in range(n_batches):
    x, y = get_batch(data, batch_size, model.block_size)
    logits = model(x)
    losses.append(F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1)).item())
  model.train()
  return float(np.mean(losses))                # a plain Python number (safe to save with torch.save)


def save_run(path, model, checkpoints, losses):
  torch.save(dict(config=model.config, vocab=uniquecharacters, checkpoints=checkpoints, losses=losses), path)

def copy_weights(model):
  return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def train_model(model, max_steps=2000, batch_size=32, learning_rate=1e-3,
                save_steps=(), run_name='music_gpt', eval_every=100, plot_every=250, data=None):
  data = train_data if data is None else data           # Part 6 trains on another text
  lossfun   = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
  losses = dict(step=[], train=[], val=[], best_step=None)
  checkpoints = {}
  t0 = time.time()

  for step in range(1, max_steps+1):
    # get input and target (shifted version of input)
    x, y = get_batch(data, batch_size, model.block_size)

    # forward pass and loss: every position predicts its next character
    logits = model(x)                                   # (batch, block_size, vocab_size)
    loss = lossfun(logits.view(-1, logits.shape[-1]), y.view(-1))

    # backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # from time to time: measure train and validation loss
    if step % eval_every == 0 or step == 1:
      losses['step'].append(step)
      losses['train'].append(estimate_loss(model, data))
      losses['val'].append(estimate_loss(model, val_data))
      # the lowest validation loss so far? keep a copy of these weights
      if losses['val'][-1] <= min(losses['val']):
        checkpoints['best'] = copy_weights(model)
        losses['best_step'] = step

    # keep a copy of the weights at some steps (and save them, in case Colab disconnects)
    if step in save_steps or step == max_steps:
      checkpoints[step] = copy_weights(model)
      save_run(os.path.join(SAVE_DIR, run_name + '.pt'), model, checkpoints, losses)

    # live plot
    if step % plot_every == 0 or step == max_steps:
      clear_output(wait=True)
      plt.figure(figsize=(8,4))
      plt.plot(losses['step'], losses['train'], label='train')
      plt.plot(losses['step'], losses['val'], label='validation')
      plt.axhline(np.log(vocab_size), color='gray', linestyle='--', label='uniform guessing')
      plt.plot(losses['best_step'], min(losses['val']), 'k*', markersize=12, label='best')
      plt.xlabel('Training step'); plt.ylabel('Loss'); plt.legend(); plt.ylim(0, 5)
      plt.title(f'step {step}/{max_steps} · val loss {losses["val"][-1]:.3f} · '
                f'perplexity {np.exp(losses["val"][-1]):.2f} · {time.time()-t0:.0f} s')
      plt.show()

  return losses, checkpoints

def encode(s): return [letter2number[c] for c in s]

@torch.no_grad()
def sample_char(ids, temperature=0.8, top_k=None):
  # the number of the next character, given the numbers of the characters so far
  context = ids[-model.block_size:]
  n = len(context)
  padded = min(model.block_size, -(-n // 128) * 128)
  x = torch.zeros(1, padded, dtype=torch.long)
  x[0, :n] = torch.tensor(context)
  logits = model(x.to(device))[0, n-1, :].float().cpu() / temperature
  if top_k is not None:
    logits[logits < torch.topk(logits, top_k).values[-1]] = -float('inf')
  return torch.multinomial(F.softmax(logits, dim=-1), 1).item()

def bass_of(tune, n=12):
  # the bass part (with figures) of the first n bars of a tune
  head, body = split_abc(tune)
  return [b.split('&')[0].strip() for b in body.replace('\n', '').replace('|]', '|').split('|')[:n]]

def header_of(tune):
  return '\n'.join(tune.split('\n')[:7]) + '\n'

def first_bars(tune, n=12):                 # the first n bars of a tune
  body = split_abc(tune)[1].replace('\n', '').replace('|]', '|').split('|')[:n]
  return header_of(tune) + '|'.join(body) + '|]'

@torch.no_grad()
def bar_candidates(text_, n_candidates, temperature=0.8, max_bar_chars=80):
  # n_candidates versions of the next melody bar, written in parallel (as one batch). None = it did not end with a bar line
  ids = torch.tensor([encode(text_)] * n_candidates)            # (n_candidates, characters)
  bars, done = [''] * n_candidates, [None] * n_candidates
  for _ in range(max_bar_chars):
    context = ids[:, -model.block_size:]
    n = context.shape[1]
    padded = min(model.block_size, -(-n // 128) * 128)          # the same padding trick as in sample_char
    x = F.pad(context, (0, padded - n))
    logits = model(x.to(device))[:, n-1, :].float().cpu() / temperature
    new = torch.multinomial(F.softmax(logits, dim=-1), 1)       # one character for every candidate
    ids = torch.cat([ids, new], dim=1)
    for i, c in enumerate(number2letter[int(k)] for k in new[:, 0]):
      if done[i] is not None: continue
      if c == '|': done[i] = True                               # the bar is complete
      elif c in '\n&': done[i] = False                          # nonsense
      else: bars[i] += c
    if None not in done: break
  return [b if d else None for b, d in zip(bars, done)]

def right_length(bass, melody, key):
  # does the melody bar last as long as the bass bar, with no unreadable character?
  if melody is None: return False
  sig = sig_acc(key_to_sharps(key))
  segs, errors = read_side(melody, sig)
  return errors == 0 and length(segs) == length(read_side(bass, sig)[0])

def fit_score(bass, melody, header):
  # % of the strong-beat notes that are in the chord. A bar with no note on a strong beat gets 0
  f = strong_fit(header + bass + ' & ' + melody + '|')[0]
  return 0 if np.isnan(f) else f

# Part 1 · Write the control into the text

In [ ]:
# download the two collections and turn them into ABC text (about 1 minute the first time)
import subprocess
DATA_FILE = os.path.join(E2D_DIR, 'baroque_abc.txt')

if not os.path.exists(DATA_FILE):
  for url, path, keep in [('https://github.com/DCMLab/corelli.git', 'corelli', ['/notes/*.tsv', '/chords/*.tsv', '/measures/*.tsv', '/metadata.tsv']),
                          ('https://github.com/juyaolongpaul/Bach_chorale_FB.git', 'bach_chorales', ['/FB_source/translated_kern/*.krn'])]:
    if not os.path.exists(path):
      subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', url, path], check=True)
      subprocess.run(['git', '-C', path, 'sparse-checkout', 'set', '--no-cone'] + keep, check=True)
  cor, skipped1 = corelli('corelli')
  cho, skipped2 = chorales('bach_chorales/FB_source/translated_kern')
  with open(DATA_FILE, 'w') as f:
    f.write('\n'.join(to_abc(x, i) for i, x in enumerate(cor + cho, 1)))

text = open(DATA_FILE).read()
tunes = text.strip().split('\n\n')       # one tune per block of text
print(f'{len(tunes)} tunes, {len(text):,} characters')

In [ ]:
# split by SONATA (and by chorale), not by movement: the movements of a sonata, and its two violins, are similar
# and the validation set must be music the model has really never seen
def group_of(tune):
  title = re.search(r'^T:(.*)$', tune, re.M).group(1)
  m = re.match(r'(.*?, no\. \d+)', title)            # 'Sonata da chiesa op. 1, no. 1'
  return m.group(1) if m else title                  # a chorale is its own group

groups_c = sorted(set(group_of(t) for t in tunes if 'Chorale' not in t))
groups_b = sorted(set(group_of(t) for t in tunes if 'Chorale' in t))
rng = random.Random(42); rng.shuffle(groups_c); rng.shuffle(groups_b)
val_groups = set(groups_c[:4]) | set(groups_b[:len(groups_b)//10])     # 4 sonatas and 10% of the chorales for validation

N_KEYS = 9   #@param {type:"integer"}
# the training tunes in N_KEYS keys (at most 9), as in Part 6 of E2d; the validation tunes as they are
train_tunes = [t for t in tunes if group_of(t) not in val_groups]
val_tunes   = [t for t in tunes if group_of(t) in val_groups]
train_tunes = [transpose(t.strip(), f).strip() for t in train_tunes for f in nearby_keys(t, N_KEYS)]
random.Random(0).shuffle(train_tunes)

In [ ]:
def note_counts(tune):
  # how many notes START in each melody bar (a note tied from before does not start; rests do not count)
  counts, tied = [], False
  for b in read_bars(tune)[1]:
    n = 0
    for d, p, tie, f in (b[1] if b is not None else []):
      if p is not None and not tied: n += 1
      tied = tie and p is not None
    counts.append(n)
  return counts

def note_levels(tune):
  # the control: always ONE digit, 9 = 9 or more (bars with more notes are rare)
  return [min(n, 9) for n in note_counts(tune)]

def add_control(tune):
  # write ~n after the & of every bar
  head = header_of(tune)
  levels = iter(note_levels(tune))
  return head + re.sub(r' & ', lambda m: f' & ~{next(levels)} ', tune[len(head):])

train_control = [add_control(t) for t in train_tunes]
val_control   = [add_control(t) for t in val_tunes]
train_text = '\n\n'.join(train_control) + '\n\n'
val_text   = '\n\n'.join(val_control) + '\n\n'
print(val_control[0][:400])
print(f'\ntrain: {len(train_control):,} tunes, {len(train_text):,} characters · val: {len(val_control)} tunes, {len(val_text):,} characters')

In [ ]:
# how busy are the real bars? (the model can only learn what it has seen)
counts = [n for t in train_tunes for n in note_counts(t)]
plt.figure(figsize=(9, 3))
plt.hist(counts, bins=np.arange(-0.5, max(counts) + 1.5), rwidth=.8)
plt.xlabel('notes that start in the bar'); plt.ylabel('bars'); plt.show()
print('with one digit:', {f'~{n}': sum(min(c, 9) == n for c in counts) for n in range(10)})

❓ **Questions**
1. Why does the number come **before** the melody and not after it? What would the model learn if it came after?
2. In a first version we wrote `~1`, `~12`, `~16`… and the model mixed them up: they all start with the same character. Why is one digit easier to learn? What do we lose?
3. Why do we count the notes that *start*, and not the tied ones?

# Part 2 · Fine-tuning

We do not start from zero. The E2d model already knows bass, figures and melody; it only has to learn one new thing. We **load its weights** and continue training, with a smaller learning rate (3e-4 instead of 1e-3) so as not to break what it knows.

One problem: `~` is a **new character**. The embedding table and the output layer need one more row. We copy the old rows, and the new one starts random.

**Which characters count?** We always give the bass, so the model does not need to learn to write it. We compute the loss **only on the number and the melody**: in the target, all the other characters become `-100`, a value that `CrossEntropyLoss` ignores. The model still *reads* the bass; it is just not graded on it.

The **best checkpoint** is chosen with this same loss. In our first version the loss counted everything, and the bass and the figures (which overfit early) hid that the melody was still improving.

The loss cannot be compared with E2d.

In [ ]:
def enlarge(checkpoint, new_vocab, dropout):
  # the E2d model with room for the new characters (added at the end of the vocabulary)
  old = checkpoint['checkpoints']['best']
  config = dict(checkpoint['config'], vocab_size=len(new_vocab), dropout=dropout)
  model = MusicGPT(**config)
  new = model.state_dict()
  for name, weights in old.items():
    if new[name].shape == weights.shape: new[name] = weights
    else: new[name][:len(weights)] = weights        # tok_emb.weight, head.weight, head.bias: one row more
  model.load_state_dict(new)
  return model.to(device)

base = torch.load(os.path.join(E2D_DIR, 'e2d_baroque_gpt_6keys.pt'), map_location='cpu')
uniquecharacters = list(base['vocab']) + ['~']
vocab_size = len(uniquecharacters)
number2letter = dict(enumerate(uniquecharacters))
letter2number = {l: i for i, l in number2letter.items()}

def make_data(text):
  # the input (all characters) and the target (-100 = ignored, except from ~ to the bar line)
  inputs  = torch.tensor(encode(text), dtype=torch.long)
  targets = torch.full_like(inputs, -100)
  for m in re.finditer(r'~[^|]*\|', text):              # '~4 f2 g2 a4|'
    targets[m.start():m.end()] = inputs[m.start():m.end()]
  return inputs, targets

train_data = make_data(train_text)
val_data   = make_data(val_text)
block_size, batch_size = base['config']['block_size'], 32
print(f"E2d vocabulary: {len(base['vocab'])} characters → E3: {vocab_size}")
print(f'{100 * (train_data[1] >= 0).float().mean():.0f}% of the characters count in the loss')

def get_batch(data, batch_size=32, block_size=512):
  # like in E2d, but the target comes from its own tensor
  inputs, targets = data
  ix = torch.randint(len(inputs) - block_size - 1, (batch_size,))
  x = torch.stack([inputs [i   : i+block_size  ] for i in ix])
  y = torch.stack([targets[i+1 : i+block_size+1] for i in ix])
  return x.to(device), y.to(device)

x, y = get_batch(train_data, 1, 100)
print('input: ', repr(''.join(number2letter[int(i)] for i in x[0])))
print('target:', repr(''.join(number2letter[int(i)] if i >= 0 else '.' for i in y[0])))     # . = ignored

In [ ]:
#@title Fine-tune (time not measured yet)
TRAIN = True         #@param {type:"boolean"}
FROM_SCRATCH = False #@param {type:"boolean"}
# FROM_SCRATCH = True: start from random weights, to compare (use learning_rate = 1e-3)
max_steps = 6000     #@param {type:"integer"}
learning_rate = 3e-4 #@param {type:"number"}
RUN_NAME = 'e3_rhythm_gpt_v2'

if TRAIN:
  torch.manual_seed(0)
  model = enlarge(base, uniquecharacters, dropout=0.2)
  if FROM_SCRATCH: model = MusicGPT(**model.config).to(device)
  losses, checkpoints = train_model(model, max_steps=max_steps, batch_size=batch_size, learning_rate=learning_rate,
                                    save_steps=[500, 2000, max_steps], run_name=RUN_NAME, plot_every=250)
else:
  run = torch.load(os.path.join(SAVE_DIR, RUN_NAME + '.pt'), map_location='cpu')
  losses, checkpoints = run['losses'], run['checkpoints']
  model = MusicGPT(**run['config']).to(device)
model.load_state_dict(checkpoints['best'])

In [ ]:
# check out the losses
plt.figure(figsize=(8,4))
plt.plot(losses['step'], losses['train'], label='train')
plt.plot(losses['step'], losses['val'], label='validation')
for s in sorted(k for k in checkpoints if k != 'best'):
  plt.axvline(s, color='k', alpha=.2)
plt.plot(losses['best_step'], min(losses['val']), 'k*', markersize=12, label='best')
plt.xlabel('Training step'); plt.ylabel('Loss'); plt.legend()
plt.show()
print(f'final validation loss {losses["val"][-1]:.3f} → perplexity {np.exp(losses["val"][-1]):.2f} '
      f'(uniform guessing: {vocab_size})')
print(f'best validation loss {min(losses["val"]):.3f} at step {losses["best_step"]}: checkpoints["best"]')

# Part 3 · Calm or lively?

`generate_control` writes the bass **and** the control of every bar; the model writes the melody. With `None` the model chooses the number itself. (The model was not graded on the bass, so it can no longer write everything alone, as in Part 4 of E2d.)

In [ ]:
def generate_control(bass_bars, header, control=None, temperature=0.8, max_bar_chars=100):
  # control: None (the model decides), a number for all bars, or a list of numbers (repeated if it is too short)
  model.eval()
  text_ = header
  for i, b in enumerate(bass_bars):
    n = control[i % len(control)] if isinstance(control, (list, tuple)) else control
    text_ += b.strip() + ' & ' + ('' if n is None else f'~{n} ')
    ids = encode(text_)
    for _ in range(max_bar_chars):
      c = number2letter[sample_char(ids, temperature)]
      if c in '|\n&': break
      text_ += c; ids.append(letter2number[c])
    text_ += '|'
    if (i + 1) % 4 == 0: text_ += '\n'
  return text_.rstrip('\n')[:-1] + '|]'

real = next(t for t in val_tunes if 'op. 3, no. 12, 6. Allegro (violin 1)' in t)      # an unseen bass; Corelli's melody has from 4 to 15 notes per bar
for control in [2, 5, 9]:
  torch.manual_seed(5)
  tune = generate_control(bass_of(real), header_of(real), control)
  print(f'================  ~{control}  ================   notes obtained:', note_counts(tune))
  report(tune); show_abc(tune, midi_name=f'control{control}')

In [ ]:
# a curve: the piece gets livelier, then calms down
curve = [2, 2, 4, 4, 6, 8, 9, 9, 8, 6, 4, 1]
torch.manual_seed(5)
tune = generate_control(bass_of(real), header_of(real), curve)
print('asked:   ', curve); print('obtained:', note_counts(tune))
show_abc(tune, midi_name='curve')

# Corelli's own numbers, then Corelli
asked = note_levels(first_bars(real))
torch.manual_seed(5)
tune = generate_control(bass_of(real), header_of(real), asked)
print('\nMODEL with the numbers of Corelli'); print('asked:   ', asked); print('obtained:', note_counts(tune)); report(tune); show_abc(tune, midi_name='corelli_numbers')
print('CORELLI'); show_abc(first_bars(real), midi_name='corelli')

In [ ]:
#@title Your own bass and your own curve
bass = ['D,2 D,2 C,2 B,,2', '"_6"B,,2 A,,2 G,,2 "_7"A,,2', 'D,2 D,2 C,2 B,,2', '"_6"B,,2 A,,2 "_4 2"G,,2 "_#"A,,2']
curve = [2, 4, 4, 6, 6, 8, 8, 9, 9, 8, 4, 2]
meter = '4/4'    #@param ['4/4', '3/4', '3/2']
key = 'D'        #@param ['F', 'C', 'G', 'D', 'Gdor', 'Am', 'Dm', 'Bm', 'Em']
kind = 'allegro' #@param ['allegro', 'grave', 'adagio', 'vivace', 'sarabanda', 'chorale']
tune = generate_control(bass * 3, make_prompt(kind=kind, meter=meter, key=key), curve)
print('asked:   ', curve); print('obtained:', note_counts(tune))
report(tune); show_abc(tune, midi_name='my_curve')

❓ **Questions**
4. Is `~9` livelier, or only faster? Does it still fit the bass?
5. Ask for `~0` in every bar, or `~9` in a chorale. These are **out of distribution**. What happens, and why?
6. Change `kind` to `grave` and ask for `~9`. The header and the control disagree. Which one wins?

# Part 4 · Measure the control

A control is good if the model **obeys** it and the music does not get **worse**. We ask for the same number in every bar of unseen basses, and compare with what we get.

In [ ]:
tests = [t for t in val_tunes if '\nM:4/4\n' in t and 'Chorale' not in t and '(violin 1)' in t][:8]
requests = [1, 2, 3, 4, 5, 6, 7, 8, 9]
rows = []
def measure(who, tune, asked):
  f = strong_fit(tune); got = note_levels(tune)
  for a, g in zip(asked, got): rows.append(dict(who=who, asked=a, obtained=g, error=abs(g - a), exact=100*(g == a), near=100*(abs(g - a) <= 1)))
  return dict(who=who, fit=f[0], bars_ok=100*check_abc(tune)['bar_ok'], notes=np.mean(note_counts(tune)))

music = []
for i, t in enumerate(tests):
  for n in requests + [None, 'Corelli']:
    torch.manual_seed(100 + i)
    real_counts = note_levels(first_bars(t))
    control = real_counts if n == 'Corelli' else n
    x = generate_control(bass_of(t), header_of(t), control)
    if n is None: asked = [int(k) for k in re.findall(r'~(\d+)', x)]        # what the model asked of itself
    else: asked = control if n == 'Corelli' else [n] * len(bass_of(t))
    music.append(measure('the model decides' if n is None else 'numbers of Corelli' if n == 'Corelli' else f'~{n}', x, asked))
  music.append(measure('real', first_bars(t), real_counts))
df = pd.DataFrame(rows)
fixed = df[df.who.str.startswith('~')]

plt.figure(figsize=(6, 4))
plt.plot([0, 9], [0, 9], 'k--', label='perfect')
g = fixed.groupby('asked').obtained
plt.errorbar(g.mean().index, g.mean(), yerr=g.std(), fmt='o-', label='obtained (mean ± sd)')
plt.xlabel('notes asked (9 = 9 or more)'); plt.ylabel('notes obtained (9 = 9 or more)'); plt.legend(); plt.show()
print('OBEDIENCE'); print(df.groupby('who')[['asked', 'obtained', 'error', 'exact', 'near']].mean().round(1))
print('\nMUSIC'); print(pd.DataFrame(music).groupby('who').mean().round(1))
print('correlation asked / obtained:', round(fixed.asked.corr(fixed.obtained), 2), '(0 = ignores the control, 1 = follows it)')

In [ ]:
# the control experiment: the E2d model (no control token) on the same basses. Does E3 get closer to Corelli's notes per bar?
def without_control():
  # generate_over of E2d, with the E2d model and its own vocabulary
  global model, letter2number, number2letter
  keep = model, letter2number, number2letter
  model = MusicGPT(**base['config']).to(device); model.load_state_dict(base['checkpoints']['best'])
  number2letter = dict(enumerate(base['vocab'])); letter2number = {l: i for i, l in number2letter.items()}
  out = []
  for i, t in enumerate(tests):
    torch.manual_seed(100 + i)
    out.append(generate_control(bass_of(t), header_of(t), None))
  model, letter2number, number2letter = keep
  return out
for x in without_control(): music.append(dict(who='E2d (no control)', fit=strong_fit(x)[0], bars_ok=100*check_abc(x)['bar_ok'], notes=np.mean(note_counts(x))))
m = pd.DataFrame(music).groupby('who').mean().round(1)
print(m.loc[['E2d (no control)', 'the model decides', 'numbers of Corelli', 'real']])

❓ **Questions**
7. Where does the model obey well, and where not? Compare with the histogram of Part 1.
8. What does the control cost? Look at `fit` and `bars_ok` as the request grows.
9. Is "numbers of Corelli" closer to the real music than "E2d (no control)"? In notes per bar? In how it sounds?
10. Is the model **counting**? How could you test if it understands "8 notes" or only "lively"?

# ★ Extra · Control + candidate selection

The control says what we want; selection throws away what does not obey. For each bar we write 8 versions, discard those with the wrong length, and keep the one closest to the number; among equals, the one that best fits the bass.

In [ ]:
def generate_control_select(bass_bars, header, control, n_candidates=8, temperature=0.8):
  model.eval()
  key = split_abc(header)[0]['K']
  text_ = header
  for i, b in enumerate(bass_bars):
    b = b.strip(); n = control[i % len(control)] if isinstance(control, (list, tuple)) else control
    start = text_ + b + f' & ~{n} '
    cands = bar_candidates(start, n_candidates, temperature)
    good = [m for m in cands if right_length(b, m, key)]
    obtained = lambda m: note_levels(start + m + '|')[-1]
    if good: best = min(good, key=lambda m: (abs(obtained(m) - n), -fit_score(b, m, header)))
    else:    best = next((m for m in cands if m is not None), 'z')
    text_ = start + best + '|'
    if (i + 1) % 4 == 0: text_ += '\n'
  return text_.rstrip('\n')[:-1] + '|]'

curve = [2, 2, 4, 4, 6, 8, 9, 9, 8, 6, 4, 1]
torch.manual_seed(5)
tune = generate_control_select(bass_of(real), header_of(real), curve)
print('asked:   ', curve); print('obtained:', note_counts(tune))
report(tune); show_abc(tune, midi_name='control_select')

---
### Teacher notes (E3, rhythm control, version 2)
* **What it is.** The E2d text with `~n` after every `&` (n = melody notes that start in the bar, one digit, 9 = 9 or more). Fine-tunes the second E2d checkpoint: 6,000 steps, learning rate 3e-4, dropout 0.2, 9 keys, same validation tunes.
* **Changes from version 1** (25 Nov 2026; run `e3_rhythm_gpt.pt`, correlation asked/obtained 0.12 with its "best" checkpoint, 0.31 at step 1,500): one digit instead of exact numbers; loss only on the number and the melody; "best" chosen by that same loss; 6,000 steps instead of 1,500. See `E2_research_scripts/baroque_check/e3_control_eval__2026-11-25.txt`.
* **Without a number** the model writes `~n` itself. It cannot write the bass any more.
* **Version 2 is not trained or measured.** Only short code checks were run.
* Helpers 1–3 are copied from E2d; `read_bars` ignores the control token.